In [0]:
%run ./00_config

In [0]:
%run ./01_core_functions

In [0]:
control_df = (
    spark.table(CONTROL_TABLE)
    .filter(F.col("is_active") == 1)
)

In [0]:
control_rows = control_df.collect()

In [0]:
for row in control_rows:

    catalog_name = row["catalog_name"]
    schema_name = row["schema_name"]
    table_name = row["table_name"]

    primary_key_columns = (
        row["primary_key_columns"]
    )

    last_processed_version = (
        row["last_processed_version"]
    )

    # Handle NULL just in case
    if last_processed_version is None:
        last_processed_version = 1

    try:

        processed_version = process_table(
            catalog_name,
            schema_name,
            table_name,
            primary_key_columns,
            last_processed_version
        )

        print(
            f"Successfully processed "
            f"{catalog_name}.{schema_name}.{table_name}"
        )

    except Exception as e:

        print(
            f"ERROR processing "
            f"{catalog_name}.{schema_name}.{table_name}"
        )

        print(str(e))

        raise

In [0]:
for row in control_rows:
    catalog_name = row["catalog_name"]
    schema_name = row["schema_name"]
    table_name = row["table_name"]

    primary_key_columns = (
        row["primary_key_columns"]
    )

    last_processed_version = (
        row["last_processed_version"]
    )

    if last_processed_version is None:
        last_processed_version = 1

    try:

        # -----------------------------------------
        # Process source table
        # -----------------------------------------

        processed_version = process_table(
            catalog_name,
            schema_name,
            table_name,
            primary_key_columns,
            last_processed_version
        )


        # -----------------------------------------
        # Update control table
        # -----------------------------------------

        spark.sql(f"""
            UPDATE {CONTROL_TABLE}
            SET
                last_processed_version = {processed_version},
                updated_at = current_timestamp()
            WHERE
                catalog_name = '{catalog_name}'
                AND schema_name = '{schema_name}'
                AND table_name = '{table_name}'
        """)


        print(
            f"Updated watermark for "
            f"{catalog_name}.{schema_name}.{table_name} "
            f"to version {processed_version}"
        )


    except Exception as e:

        print(
            f"FAILED: "
            f"{catalog_name}.{schema_name}.{table_name}"
        )

        print(str(e))

        raise